### portmy database: stocks table
### stock database: buy, price tables

In [1]:
import pandas as pd
import numpy as np
from datetime import date, timedelta
from sqlalchemy import create_engine
from pandas.tseries.offsets import BDay

engine = create_engine("sqlite:///c:\\ruby\\portmy\\db\\development.sqlite3")
conmy = engine.connect()
conmy = engine.connect()
engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

pd.set_option('display.float_format','{:,.2f}'.format)

data_path = "../data/"
csv_path = "\\Users\\User\\iCloudDrive\\"
box_path = "\\Users\\User\\Dropbox\\"

today = date.today()
today

datetime.date(2022, 12, 24)

### Process after last business day, yesterday must be last business day

In [2]:
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(1)
yesterday = today - num_business_days
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2022-12-24
yesterday: 2022-12-23 00:00:00


In [3]:
yesterday = yesterday.date()
week_ago = yesterday -timedelta(days = 7)
print(f'a week ago: {week_ago}')
print(f'yesterday: {yesterday}')

a week ago: 2022-12-16
yesterday: 2022-12-23


### Restart & Run All Cells

### This process affects only already in port stocks. To highlight price changes in percent.

In [5]:
cols = 'name price_d price_w percent perform mrkt '.split()

format_dict = {
    'qty':'{:,}',    
    'price':'{:.2f}','price_d':'{:.2f}','price_w':'{:.2f}',
    'max_price':'{:.2f}','min_price':'{:.2f}',
    'maxp':'{:.2f}','minp':'{:.2f}','opnp':'{:.2f}',    
    'pct':'{:,.2f}%','percent':'{:,.2f}%',
    
    'pe':'{:.2f}','pbv':'{:.2f}',
    'paid_up':'{:,.2f}','market_cap':'{:,.2f}',
    'daily_volume':'{:,.2f}','beta':'{:,.2f}', 
    'dly_vol':'{:,.2f}', 
}

In [6]:
sql = """
SELECT name, date, price 
FROM buy 
ORDER BY date DESC 
LIMIT 1
"""
df_buy = pd.read_sql(sql, const)
df_buy.style.format(format_dict)

,name,date,price
0,GFPT,2022-12-21,12.50


In [7]:
names = df_buy['name']
name = names.to_string(index=False)

sql = '''
SELECT * FROM stocks WHERE name = '%s'
'''
sql = sql % name
print(sql)

df_stocks = pd.read_sql(sql, conmy)
df_stocks.style.format(format_dict) 


SELECT * FROM stocks WHERE name = 'GFPT'



,id,name,market,price,max_price,min_price,pe,pbv,paid_up,market_cap,daily_volume,beta,ticker_id,created_at,updated_at
0,187,GFPT,sSET / SETTHSI / SETWB,12.70,18.70,11.80,9.67,0.99,"1,253.82","15,923.53",101.78,0.60,187,2017-07-23 07:24:45.610475,2022-12-24 04:43:54.483196


In [8]:
sql = '''
SELECT * FROM price WHERE name = "%s" ORDER BY date DESC LIMIT 5
'''
sql = sql % name
print(sql)

df_price = pd.read_sql(sql, const)
df_price.style.format(format_dict)


SELECT * FROM price WHERE name = "GFPT" ORDER BY date DESC LIMIT 5



,name,date,price,maxp,minp,qty,opnp
0,GFPT,2022-12-23,12.70,12.80,12.40,"3,325,441",12.70
1,GFPT,2022-12-22,12.60,12.70,12.40,"2,246,872",12.40
2,GFPT,2022-12-21,12.40,13.20,12.30,"12,130,306",13.10
3,GFPT,2022-12-20,13.10,13.20,12.90,"5,605,293",13.20
4,GFPT,2022-12-19,13.10,13.10,12.90,"2,043,257",12.90


In [9]:
sql = '''
SELECT name
FROM buy 
WHERE active = 1
ORDER BY name'''
df_price = pd.read_sql(sql, const)

names = df_price.name.tolist()
in_p = ", ".join(map(lambda name: "'%s'" % name, names))

sql = """
SELECT name, price AS price_d 
FROM price 
WHERE date = '%s' AND name IN (%s)
ORDER BY name, date"""
sql = sql % (yesterday, in_p)
print(sql)

df_today = pd.read_sql(sql, const)
df_today.shape[0]


SELECT name, price AS price_d 
FROM price 
WHERE date = '2022-12-23' AND name IN ('ASP', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GFPT', 'GVREIT', 'IVL', 'JASIF', 'JMART', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART')
ORDER BY name, date


26

In [10]:
sql = """
SELECT name, price AS price_w
FROM price 
WHERE date = '%s' AND name IN (%s) 
ORDER BY name"""
sql = sql % (week_ago, in_p)
print(sql)

df_wkago = pd.read_sql(sql, const)
df_wkago.shape[0]


SELECT name, price AS price_w
FROM price 
WHERE date = '2022-12-16' AND name IN ('ASP', 'BCH', 'CPNREIT', 'DCC', 'DIF', 'GFPT', 'GVREIT', 'IVL', 'JASIF', 'JMART', 'KCE', 'MAKRO', 'MCS', 'NER', 'ORI', 'PTTGC', 'RCL', 'SCC', 'SCCC', 'SENA', 'STA', 'SYNEX', 'TFFIF', 'TMT', 'WHAIR', 'WHART') 
ORDER BY name


26

In [11]:
trend = pd.merge(df_today, df_wkago, on='name',how='inner')
trend.shape

(26, 3)

In [12]:
def performance(vals):
    price_d, price_w = vals
    if (price_d > price_w):
        return 'Better'
    elif (price_d < price_w):
        return 'Worse'
    else:
        return 'No Change'

In [13]:
trend['percent'] = (trend.price_d-trend.price_w)/trend.price_w * 100

In [14]:
trend

,name,price_d,price_w,percent
0,ASP,2.96,2.96,0.00
1,BCH,19.80,19.80,0.00
2,CPNREIT,19.50,19.50,0.00
3,DCC,2.78,2.78,0.00
4,DIF,13.20,13.10,0.76
5,GFPT,12.70,13.00,-2.31
6,GVREIT,9.10,8.95,1.68
7,IVL,39.50,40.00,-1.25
8,JASIF,8.00,8.05,-0.62
9,JMART,39.25,40.75,-3.68


In [15]:
trend["perform"] = trend[["price_d","price_w"]].apply(performance, axis=1)

In [16]:
trend.sort_values(by=['percent'],ascending=[True]).style.format(format_dict)

,name,price_d,price_w,percent,perform
15,PTTGC,44.75,46.75,-4.28%,Worse
10,KCE,45.50,47.25,-3.70%,Worse
9,JMART,39.25,40.75,-3.68%,Worse
5,GFPT,12.70,13.00,-2.31%,Worse
21,SYNEX,15.70,15.90,-1.26%,Worse
7,IVL,39.50,40.00,-1.25%,Worse
19,SENA,3.92,3.96,-1.01%,Worse
13,NER,5.90,5.95,-0.84%,Worse
8,JASIF,8.00,8.05,-0.62%,Worse
12,MCS,8.90,8.95,-0.56%,Worse


In [17]:
trend.perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Better,42.31%
Worse,42.31%
No Change,15.38%


In [18]:
sql = '''
SELECT name, max_price AS max, min_price AS min, pe, pbv, daily_volume AS volume, beta, market
FROM stocks
'''
my_stocks = pd.read_sql(sql, conmy)
my_stocks.shape

(253, 8)

In [19]:
filters = [
   (my_stocks.market.str.contains('SET50')),
   (my_stocks.market.str.contains('SET100')),
   (my_stocks.market.str.contains('mai'))]
values = ['SET50','SET100','mai']
my_stocks["mrkt"] = np.select(filters, values, default='SET999')

In [20]:
trend2 = pd.merge(trend, my_stocks, on='name', how='inner')
trend2.sort_values(['percent'],ascending=[True]).shape

(26, 13)

In [21]:
set50 = trend2.mrkt.str.contains('SET50') 
flt_set50 = trend2[set50]
flt_set50[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_d,price_w,percent,perform,mrkt
15,PTTGC,44.75,46.75,-4.28%,Worse,SET50
10,KCE,45.50,47.25,-3.70%,Worse,SET50
9,JMART,39.25,40.75,-3.68%,Worse,SET50
7,IVL,39.50,40.00,-1.25%,Worse,SET50
17,SCC,334.00,333.00,0.30%,Better,SET50


In [22]:
flt_set50.perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Worse,80.00%
Better,20.00%


In [23]:
set100 = trend2.mrkt.str.contains('SET100')
flt_set100 = trend2[set100]
flt_set100[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_d,price_w,percent,perform,mrkt
21,SYNEX,15.70,15.90,-1.26%,Worse,SET100
20,STA,19.00,19.10,-0.52%,Worse,SET100
1,BCH,19.80,19.80,0.00%,No Change,SET100
16,RCL,30.50,30.00,1.67%,Better,SET100
14,ORI,11.20,10.90,2.75%,Better,SET100


In [24]:
flt_set100[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Better,40.00%
Worse,40.00%
No Change,20.00%


In [25]:
set999 = trend2.mrkt.str.contains('SET999')
flt_set999 = trend2[set999]
flt_set999[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_d,price_w,percent,perform,mrkt
5,GFPT,12.70,13.00,-2.31%,Worse,SET999
19,SENA,3.92,3.96,-1.01%,Worse,SET999
13,NER,5.90,5.95,-0.84%,Worse,SET999
8,JASIF,8.00,8.05,-0.62%,Worse,SET999
12,MCS,8.90,8.95,-0.56%,Worse,SET999
0,ASP,2.96,2.96,0.00%,No Change,SET999
2,CPNREIT,19.50,19.50,0.00%,No Change,SET999
3,DCC,2.78,2.78,0.00%,No Change,SET999
18,SCCC,149.50,149.00,0.34%,Better,SET999
4,DIF,13.20,13.10,0.76%,Better,SET999


In [26]:
flt_set999[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
Better,50.00%
Worse,31.25%
No Change,18.75%


In [27]:
setmai = trend2.mrkt.str.contains('mai')
flt_setmai = trend2[setmai]
flt_setmai[cols].sort_values(by=['percent','name'],ascending=[True,True]).style.format(format_dict)

,name,price_d,price_w,percent,perform,mrkt


In [28]:
flt_setmai[cols].perform.value_counts(normalize=True).to_frame().style.format('{:.2%}')

,perform
